# SmolVLM-500M-Instruct — DIMER image captioning and VQA tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/smolvlm-vision-language-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/smolvlm-vision-language-pipeline/blob/main/tutorials/smolvlm_vision_language_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-HuggingFaceTB%2FSmolVLM--500M--Instruct-ffcc4d?style=flat)](https://huggingface.co/HuggingFaceTB/SmolVLM-500M-Instruct) [![Upstream](https://img.shields.io/badge/Upstream-huggingface%2Fsmollm-181717?style=flat&logo=github&logoColor=white)](https://github.com/huggingface/smollm) [![arXiv](https://img.shields.io/badge/arXiv-2504.05299-b31b1b.svg)](https://arxiv.org/abs/2504.05299)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** image + text to text generation (captioning and visual question answering) using the pinned SmolVLM-500M-Instruct weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/smolvlm_vision_language_pipeline/pipeline.py` at revision `387ab698f6cd`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `a7da5b986cb59b408707209984f360a5f4ad7e47` (~1020 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/1); edit the repository and regenerate rather than editing cells.

At inference one image and one user prompt are wrapped in the model's chat template, the processor resizes the image to a longest edge of 2048 px and splits it into 512 px tiles (64 visual tokens each), and the Idefics3-architecture model — a SigLIP-derived vision encoder feeding the SmolLM2-360M-Instruct text decoder, which is itself already a DIMER language-model profile — generates the assistant turn. Decoding is **greedy by default** (`do_sample=False`, `DECODING = "greedy"`), so a rerun on the same device, dtype and library versions reproduces the same text; `do_sample=True` is exposed for callers who want varied wording and gives up that determinism. The output is **free text with no score, no probability and no correctness signal**: the model writes fluent prose whether or not it is right — the model card's smoke run correctly named a red square but also claimed its corners touched the image edges, which they did not — so every answer must be read as a hypothesis. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. What upstream supplies is the model, processor and chat template; what the carried pipeline module adds is manifest verification, input validation and ceilings, prompt assembly, a fixed output contract with `new_tokens`/`truncated` run facts, and the `validate_inputs` and `evaluation_report` helpers.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, generate a synthetic image (or upload your own), stage and digest-verify the immutable upstream snapshot, surface the ceilings and the decoding contract and validate every prompt into one input manifest through the pipeline's own validation stage, run a captioning prompt and a question prompt through the public API with explicit generation settings, read the output contract correctly (including the truncation flag), understand from the evaluation report why the verdict is always `not-measurable` here and what labelled data a real evaluation needs, and export the answers plus provenance.

**This notebook does not demonstrate:** multi-image or video input (`MAX_IMAGES` = 1), object detection or grounding with coordinates, OCR with layout, text-only chat, batched inference, fine-tuning, or any accuracy claim. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (bfloat16 there, the dtype the checkpoint ships in); on the model card's CPU smoke the snapshot loaded in 5.7 s and 128 new tokens took 17.4 s, so the two-prompt default takes about a minute on a hosted CPU runtime. The pinned `torch==2.14.0` install and the 1.0 GB checkpoint are the largest downloads of the run.
- **Knowledge:** basic Python and PIL image handling; what greedy decoding is and why free text has no intrinsic accuracy.
- **Data:** the default sample is a synthetic image generated in code; BYOD is exactly one image file, gated off by default, any mode (converted to RGB), with both sides between 1 and `MAX_IMAGE_SIDE` = 4096 px. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded images remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `HuggingFaceTB/SmolVLM-500M-Instruct` snapshot (~1020 MB) at revision `a7da5b986cb5…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `PIL` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'smolvlm-vision-language-pipeline',
    'repository_revision': '387ab698f6cd65c9c304ffcf288b9a2531811bc1',
    'embedded_module': 'src/smolvlm_vision_language_pipeline/pipeline.py',
    'module_sha256': '785586dd03b7711f8358e50556491f5ce0186090f4370c26189220e1e6b0332d',
    'generator': 'build_notebook.py/1',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/smolvlm_vision_language_pipeline/pipeline.py` @ `387ab698f6cd`)

This cell **is** the repository's pipeline module: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the module's, byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (currently 1: the default weights directory becomes working-directory-relative because a notebook has no `__file__`). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever this cell and the module diverge, so what you run here is what the repository tests. Nothing in this cell runs a model yet.

In [ ]:
"""Image + text -> text chat generation with the pinned ``HuggingFaceTB/SmolVLM-500M-Instruct`` snapshot.

The class loads weights only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly
allowed, from the Hugging Face Hub at the pinned revision. One image and one user text turn are rendered
through the snapshot's chat template; decoding is greedy unless ``do_sample=True`` is passed.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
MODEL_REVISION = "a7da5b986cb59b408707209984f360a5f4ad7e47"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "smolvlm-500m-instruct"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "model.safetensors"
CONFIG_FILE = "config.json"

MAX_IMAGE_SIDE = 4096  # pixels; the processor resizes to longest_edge 2048 and splits into 512-px tiles
MAX_IMAGES = 1  # v0.1.0 accepts exactly one image per call
MAX_TEXT_CHARS = 2000  # characters of user prompt accepted
MAX_NEW_TOKENS = 512  # hard ceiling for `max_new_tokens`
DEFAULT_MAX_NEW_TOKENS = 128
DECODING = "greedy"  # do_sample=False by default -> deterministic on a fixed device/dtype


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def build_messages(prompt: str) -> list[dict[str, Any]]:
    """One user turn with one image placeholder followed by the text, in the snapshot chat-template shape."""
    return [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "exactly one PIL.Image.Image (any mode, converted to RGB) plus one non-empty user prompt string"
    ),
    "images": [1, MAX_IMAGES],
    "image_side_px": [1, MAX_IMAGE_SIDE],
    "prompt_chars": [1, MAX_TEXT_CHARS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "decoding": (
        f"{DECODING} by default (do_sample=False), deterministic on a fixed device and dtype; "
        "do_sample=True trades that determinism for varied wording"
    ),
    "preprocessing": (
        "image converted to RGB; the processor resizes it so the longest edge is 2048 px (aspect ratio "
        "preserved, nothing cropped) and splits it into 512-px tiles of 64 visual tokens each; the prompt "
        "is wrapped in the snapshot's chat template as one user turn (see build_messages)"
    ),
}


def _check_inputs(images: Any, prompt: Any, max_new_tokens: Any) -> list[Image.Image]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the images as a list.

    ``SmolVLMPipeline.generate`` and ``validate_inputs`` both route through this function so their
    acceptance criteria cannot diverge.
    """
    if isinstance(images, Image.Image):
        images = [images]
    if not isinstance(images, list | tuple):
        raise TypeError("images must be a PIL.Image.Image or a list of them")
    if not 1 <= len(images) <= MAX_IMAGES:
        raise ValueError(f"image count must be between 1 and MAX_IMAGES={MAX_IMAGES}, got {len(images)}")
    for image in images:
        if not isinstance(image, Image.Image):
            raise TypeError(f"each image must be a PIL.Image.Image, got {type(image).__name__}")
        width, height = image.size
        if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:
            raise ValueError(f"image side outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: {image.size}")
    if not isinstance(prompt, str):
        raise TypeError("prompt must be a str")
    if not prompt.strip():
        raise ValueError("prompt must not be empty")
    if len(prompt) > MAX_TEXT_CHARS:
        raise ValueError(f"prompt exceeds MAX_TEXT_CHARS={MAX_TEXT_CHARS}: {len(prompt)}")
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    return list(images)


def validate_inputs(
    images: Image.Image | list[Image.Image],
    prompt: str,
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    do_sample: bool = False,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``generate`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    checked = _check_inputs(images, prompt, max_new_tokens)
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per image")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[index] if names else f"image-{index}",
                "mode": image.mode,
                "size": list(image.size),
            }
            for index, image in enumerate(checked)
        ],
        "prompt": prompt,
        "prompt_chars": len(prompt),
        "generation": {
            "max_new_tokens": max_new_tokens,
            "do_sample": bool(do_sample),
            "decoding": "sampling" if do_sample else DECODING,
        },
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


_NEEDS = (
    "labelled data matched to the use and the caller's own scoring code over enough items to state a "
    "dispersion: question-answer pairs with reference answers for VQA accuracy, reference captions for a "
    "caption metric such as CIDEr, or document pages with gold answers for document-QA exact match. This "
    "repository ships no metric helper, so there is nothing to compute here."
)
_SCORE_SEMANTICS = (
    "the generated text carries no score, no probability and no correctness signal; a fluent, specific "
    "answer is not evidence that it is right. Greedy decoding makes the text reproducible on a fixed "
    "device and dtype, which is a reproducibility property, not a quality one"
)


def evaluation_report(
    result: Mapping[str, Any] | Sequence[Mapping[str, Any]],
    references: Any = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report, always ``not-measurable`` for this capability.

    ``result`` is one ``generate`` result or a sequence of them. Open-ended image-conditioned
    generation has no intrinsic correctness signal and this repository ships no metric helper, so the
    verdict is always ``not-measurable`` (EVAL9) and ``needs`` names the labelled data a real
    evaluation would require. ``references`` is accepted and echoed so a caller can record what they
    compared against by hand; supplying it does not create a metric.
    """
    results = [result] if isinstance(result, Mapping) else list(result)
    return {
        "task": "image + text -> text generation (captioning and visual question answering)",
        "score_semantics": _SCORE_SEMANTICS,
        "sample_kind": sample_kind,
        "n_answers": len(results),
        "answers": [
            {
                "prompt": item.get("prompt"),
                "new_tokens": item.get("new_tokens"),
                "truncated": item.get("truncated"),
                "generation": dict(item.get("generation") or {}),
            }
            for item in results
        ],
        "references_supplied": references if references is None else list(references),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": (
            "open-ended generated text has no intrinsic correctness signal and this repository ships no "
            "metric helper; reading the answers against what you can see is a sanity check on one "
            "sample, not a measurement"
        ),
        "needs": _NEEDS,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class SmolVLMPipeline:
    """``_runner(image, prompt, max_new_tokens, do_sample)`` returns ``{"text": str, "new_tokens": int}``."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> SmolVLMPipeline:
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor

        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        dtype = torch.bfloat16 if resolved_device.startswith("cuda") else torch.float32
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        processor = AutoProcessor.from_pretrained(location, **common)
        model = AutoModelForImageTextToText.from_pretrained(location, dtype=dtype, **common)
        model = model.eval().to(resolved_device)

        def runner(image: Image.Image, prompt: str, max_new_tokens: int, do_sample: bool) -> dict[str, Any]:
            text = processor.apply_chat_template(build_messages(prompt), add_generation_prompt=True)
            inputs = processor(text=text, images=[image], return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample)
            new_ids = generated[0, inputs["input_ids"].shape[1] :]
            decoded = processor.batch_decode(new_ids.unsqueeze(0), skip_special_tokens=True)[0]
            return {"text": decoded, "new_tokens": int(new_ids.shape[0])}

        return cls(runner, resolved_device, str(dtype).removeprefix("torch."), source)

    def _validate(self, images: Any, prompt: Any, max_new_tokens: int) -> list[Image.Image]:
        return _check_inputs(images, prompt, max_new_tokens)

    def generate(
        self,
        images: Image.Image | list[Image.Image],
        prompt: str,
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        do_sample: bool = False,
    ) -> dict[str, Any]:
        """Answer ``prompt`` about one image; ``text`` is the decoded assistant turn (no special tokens)."""
        batch = self._validate(images, prompt, max_new_tokens)
        raw = self._runner(batch[0].convert("RGB"), prompt, max_new_tokens, bool(do_sample))
        if not isinstance(raw, dict) or "text" not in raw:
            raise RuntimeError("runner must return a dict with 'text'")
        new_tokens = int(raw.get("new_tokens", 0))
        return {
            "text": str(raw["text"]).strip(),
            "prompt": prompt,
            "image_size": list(batch[0].size),
            "new_tokens": new_tokens,
            "truncated": new_tokens >= max_new_tokens,
            "generation": {
                "max_new_tokens": max_new_tokens,
                "do_sample": bool(do_sample),
                "decoding": "sampling" if do_sample else DECODING,
            },
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `13`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `a7da5b986cb5…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `SmolVLMPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "smolvlm-500m-instruct",
  "modelId": "HuggingFaceTB/SmolVLM-500M-Instruct",
  "revision": "a7da5b986cb59b408707209984f360a5f4ad7e47",
  "files": [
    {
      "path": "README.md",
      "bytes": 9918,
      "sha256": "8465c1a46b0db5d5d0d0945df04032d1b405bb2fa12cdab3da1652bd75fca4b9"
    },
    {
      "path": "added_tokens.json",
      "bytes": 4739,
      "sha256": "74135b8664b56088c0006f1c8e848d79a8eba003411f72ebf1dc2ee96227be3a"
    },
    {
      "path": "chat_template.json",
      "bytes": 429,
      "sha256": "a68ad1a42681ae44eacd109ff8dd56a840f761c03d72f4ff4c515d092f882168"
    },
    {
      "path": "config.json",
      "bytes": 7339,
      "sha256": "daacbbca6af3c34e50466c72aed2df4553a08084f6a0551d4ae420a241cb66c6"
    },
    {
      "path": "generation_config.json",
      "bytes": 136,
      "sha256": "067a2a54e5f87162ecac6e0e911cc4665fc8f7f3324794ecbac0f76badb56636"
    },
    {
      "path": "merges.txt",
      "bytes": 466391,
      "sha256": "0b54e8aa4e53d5383e2e4bc635a56b43f9647f7b13832d5d9ecd8f82dac4f510"
    },
    {
      "path": "model.safetensors",
      "bytes": 1015025832,
      "sha256": "d05b567eeaf534e83d375551f068ed57b5f52d37c657197f644af5ef9db091a2"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 486,
      "sha256": "6cb6e36d6fcb88ca1502c4a26750715dc3e7dedddc9a8f17b27d8d167d1457e7"
    },
    {
      "path": "processor_config.json",
      "bytes": 68,
      "sha256": "e7bff42da73ae9eec9042ef20e066e11f1ee20f025358ff79131e3c0fb549b46"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 1069,
      "sha256": "aa0ff906077086dfa9734a7f97f68c825877a48f9468807be65504495cdeef09"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3548256,
      "sha256": "5ece781dc8d2b2f3e2f289ca0ae50b17cfc27dd27bfe7971bb8241e0b964331a"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 28249,
      "sha256": "36c6fd44d07d10fd8180ee6b46dcccf69fb7c06753968ff0d7e17b8bfe17b777"
    },
    {
      "path": "vocab.json",
      "bytes": 800662,
      "sha256": "82b84012e3add4d01d12ba14442026e49b8cbbaead1f79ecf3d919784f82dc79"
    }
  ],
  "totalBytes": 1019893574
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
print({'verified_files': [entry['path'] for entry in snapshot.get('files', [])]})
pipe = SmolVLMPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Generate the synthetic sample or optional BYOD

The default sample is **synthetic**: a 384 × 384 white canvas drawn in this cell with a filled red square in the upper left and a filled blue circle in the lower right — so it needs no download, contains no personal data, and is reproducible from code (no randomness, no seed; its pixel SHA-256 is printed and exported). Two prompts are asked about it: a captioning prompt and a counting question. The drawing has **no reference answers** the notebook asserts, so every answer it produces is smoke/sanity evidence that the code path works — you can judge the answers by eye, but that is a reading, not a measurement, and the model card's smoke run shows how a fluent answer can be wrong in detail.

BYOD is optional and disabled by default. Edit the `PROMPTS` form field (one prompt per `|`, each at most `MAX_TEXT_CHARS` characters) to ask your own questions; the upload stays inside this runtime. Nothing is validated in this cell — the next section hands every prompt to the pipeline's own validation stage, which is the only checker.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
PROMPTS = 'Describe this image in one sentence. | How many shapes are in the image, and what colour is each one?'  # @param {type:"string"}
MAX_NEW_TOKENS_PER_PROMPT = 96  # @param {type:"integer"}

prompts = [prompt.strip() for prompt in PROMPTS.split('|') if prompt.strip()]
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f'upload exactly one image, got {len(uploaded)}')
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    sample_kind = 'BYOD upload'
else:
    # Deterministic drawing: a red square and a blue circle on white.
    image = Image.new('RGB', (384, 384), (255, 255, 255))
    draw = ImageDraw.Draw(image)
    draw.rectangle((48, 48, 176, 176), fill=(220, 30, 30))
    draw.ellipse((208, 208, 336, 336), fill=(30, 60, 220))
    image_name = 'synthetic_square_circle_384'
    sample_kind = 'synthetic (drawn in this cell)'

sample_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample': image_name, 'sample_kind': sample_kind, 'mode': image.mode, 'size': image.size, 'prompts': prompts, 'max_new_tokens_per_prompt': MAX_NEW_TOKENS_PER_PROMPT, 'pixel_sha256': sample_sha256})

## 5. Validate every prompt → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `generate` applies — exactly one image (`MAX_IMAGES` = 1) of type `PIL.Image.Image` with both sides in 1..`MAX_IMAGE_SIDE` px, a non-empty prompt of at most `MAX_TEXT_CHARS` characters, and `max_new_tokens` in 1..`MAX_NEW_TOKENS` — and returns an **input manifest** naming the schema and ceilings, the image's observed mode and size, the prompt and its length, and the exact generation settings (including whether decoding is `greedy` or `sampling`). One prompt is validated per entry, and the combined manifest — a top-level record plus one sub-manifest per prompt under `prompts` — is written to `outputs/smolvlm_vision_language_input_manifest.json`. To show what rejection looks like, the cell also validates an over-long prompt and records the pipeline's own error message as a finding.

**What the pipeline changes about your image:** the processor resizes it so the longest edge is 2048 px (aspect ratio preserved) and splits it into 512 px tiles, each becoming 64 visual tokens; nothing is cropped away. An answer that uses the whole `max_new_tokens` budget is reported as `truncated` — a cut-off answer, not a complete one.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MAX_IMAGES': MAX_IMAGES, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DECODING': DECODING}})
manifests = [validate_inputs(image, prompt, max_new_tokens=MAX_NEW_TOKENS_PER_PROMPT, do_sample=False, names=[image_name]) for prompt in prompts]
input_manifest = {**manifests[0], 'prompt': None, 'prompt_chars': None, 'n_prompts': len(manifests), 'findings': [], 'prompts': manifests}
# Demonstrate rejection on a prompt that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(image, 'x' * (MAX_TEXT_CHARS + 1))
except ValueError as exc:
    input_manifest['findings'].append({'input': 'over-long-prompt-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/smolvlm_vision_language_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Generate answers and interpret them

Each call to `generate(images, prompt, *, max_new_tokens=..., do_sample=False)` returns `text` (the decoded assistant turn, special tokens stripped), the `prompt`, `image_size`, `new_tokens`, `truncated` (true when the answer used the whole `max_new_tokens` budget and was cut off), the `generation` settings actually used (`max_new_tokens`, `do_sample`, `decoding`), device, dtype, source and model identity. **Output semantics:** `text` is free-form generated language with **no score and no correctness signal**; a fluent, specific answer is not evidence that it is right. With greedy decoding the same inputs reproduce the same text on a fixed device, dtype and library version — that is a reproducibility property, not a quality one. The checks below are plumbing checks (text returned, settings as requested), and the truncation flag tells you whether an answer was cut. The printed seconds are measured on this runtime for this image and include the first-call warm-up.

In [ ]:
import time

results = []
answers = []
for prompt in prompts:
    started = time.perf_counter()
    result = pipe.generate(image, prompt, max_new_tokens=MAX_NEW_TOKENS_PER_PROMPT, do_sample=False)
    results.append(result)
    answers.append({'prompt': prompt, 'text': result['text'], 'new_tokens': result['new_tokens'], 'truncated': result['truncated'], 'generation': result['generation'], 'seconds': round(time.perf_counter() - started, 3)})
    print({'prompt': prompt, 'seconds': answers[-1]['seconds'], 'new_tokens': result['new_tokens'], 'truncated': result['truncated'], 'generation': result['generation']})
    print('answer:', result['text'])
checks = {
    'one_answer_per_prompt': len(answers) == len(prompts),
    'answers_are_text': all(isinstance(a['text'], str) and a['text'].strip() for a in answers),
    'greedy_as_requested': all(a['generation']['do_sample'] is False and a['generation']['decoding'] == DECODING for a in answers),
    'budget_as_requested': all(a['generation']['max_new_tokens'] == MAX_NEW_TOKENS_PER_PROMPT for a in answers),
}
if not all(checks.values()):
    raise RuntimeError(f'generate output failed a sanity check: {checks}')
print({'checks': checks, 'any_truncated': any(a['truncated'] for a in answers)})

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report — and for this capability its verdict is **always `not-measurable`**. Open-ended image-conditioned generation has no intrinsic correctness signal, and this repository ships **no metric helper**, so there is nothing to compute and nothing is invented. The report records what was generated (each prompt, its `new_tokens` and whether it was `truncated`, and the generation settings), states the score semantics, and names what a real evaluation would need: labelled data matched to the use — question-answer pairs with reference answers for VQA accuracy, reference captions for a caption metric such as CIDEr, or document pages with gold answers for document-QA exact match — plus the caller's own scoring code over enough items to state a dispersion. You can compare the answers with what you see (a red square and a blue circle), but that reading is a sanity check on one drawing, not a measurement. The report is written to `outputs/smolvlm_vision_language_evaluation_report.json`.

In [ ]:
report = evaluation_report(results, sample_kind=sample_kind)
with open('outputs/smolvlm_vision_language_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No metric is reported: free-text answers have no intrinsic correctness signal and the repository ships no metric helper.')

## 8. Export answers and provenance

Two files are written under `outputs/`: `smolvlm_vision_language_result.json` with an `answers` list carrying, per prompt, the prompt, the generated text, `new_tokens`, `truncated`, the generation settings and seconds (so every answer maps back to its prompt and the image), the plumbing checks, the evaluation report, the input manifest, the ceilings in force, the sample identity (name, kind, size, pixel digest), the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, Pillow, device, dtype); and `smolvlm_vision_language_answers.csv` with explicit `index`, `prompt`, `new_tokens`, `truncated` and `text` columns so prompt order survives downstream use. No credentials are involved in any step, so none can reach the export.

In [ ]:
import csv

payload = {
    'answers': [{'index': index, **answer} for index, answer in enumerate(answers)],
    'sanity_checks': checks,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'ceilings': {'MAX_IMAGES': MAX_IMAGES, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DECODING': DECODING},
    'sample': {'name': image_name, 'kind': sample_kind, 'width': image.width, 'height': image.height, 'pixel_sha256': sample_sha256},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'pillow': PIL.__version__,
        'device': pipe.device,
        'dtype': pipe.dtype,
        'source': pipe.source,
    },
}
with open('outputs/smolvlm_vision_language_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/smolvlm_vision_language_answers.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['index', 'prompt', 'new_tokens', 'truncated', 'text'])
    for index, answer in enumerate(answers):
        writer.writerow([index, answer['prompt'], answer['new_tokens'], answer['truncated'], answer['text']])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The answers are generated language, not measurements: they carry no score, no probability and no signal of correctness, and the model can state details that are not in the image — the model card's smoke run named a red square correctly and then invented that its corners touched the edges. On the synthetic drawing the answers are plumbing evidence only; the evaluation report is `not-measurable` because no metric exists without labelled question-answer pairs or reference captions, and a real evaluation needs such a set in your domain plus your own scoring code. The pipeline accepts one image and one prompt per call, resizes the longest edge to 2048 px and tiles it, caps answers at `MAX_NEW_TOKENS` (a `truncated` answer is cut, not complete), and exposes no grounding coordinates, no OCR layout, no text-only chat and no batching. Greedy decoding is deterministic on a fixed device and dtype (CPU float32 and CUDA bfloat16 can produce different text); `do_sample=True` trades that determinism for varied wording. The text decoder, SmolLM2-360M-Instruct, is the same model behind the DIMER language-model profile of that name, so its language-side limits (small model, English-centric instruction tuning) apply here too.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model snapshot, validate the demonstrated input against the enforced ceilings, execute the public pipeline path with explicit greedy generation settings, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, captioning or VQA accuracy on any domain, freedom from hallucination, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file is incomplete or altered — delete it from `weights/smolvlm-500m-instruct/` and rerun Section 3. A `ValueError` naming `MAX_IMAGE_SIDE`, `MAX_TEXT_CHARS` or the image count in Section 5: fix the BYOD input or the `PROMPTS` field and rerun from Section 4. `truncated: True` on an answer: raise `MAX_NEW_TOKENS_PER_PROMPT` in Section 4 (up to `MAX_NEW_TOKENS`). Slow generation on a CPU runtime is expected (about 0.14 s per token on the card's machine).

**Next experiments.** Upload a photograph and ask a question whose answer you know, then ask a question whose answer is not in the image and watch whether the model declines or invents one; rerun the default prompts with `do_sample=True` to see wording vary while the greedy run stays fixed; run the same prompts on a CUDA runtime and diff the bfloat16 answers against the CPU float32 ones. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: https://github.com/kurtvalcorza/smolvlm-vision-language-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/smolvlm-vision-language-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/smolvlm-vision-language-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/HuggingFaceTB/SmolVLM-500M-Instruct
- Upstream code: https://github.com/huggingface/smollm
- SmolVLM paper: https://arxiv.org/abs/2504.05299
- Text decoder (DIMER language-model profile): https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct